<a href="https://colab.research.google.com/github/carlosdouglas1313/processos_stk/blob/master/diferenca_pacs_ppm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from datetime import datetime, timezone, timedelta
import numpy as np

# Import the spreadsheets
df_pacs = pd.read_excel('/content/extracao_pacs.xlsx')
df_ppm = pd.read_excel('/content/extracao_ppm.xlsx')

# Clean the columns
df_pacs['Atendimento'] = df_pacs['Atendimento'].astype(str).str.replace(r'\D+', '', regex=True)
df_ppm['N° Cliente'] = df_ppm['N° Cliente'].astype(str).str.replace(r'\D+', '', regex=True)

df_pacs['Atendimento'] = pd.to_numeric(df_pacs['Atendimento'], errors='coerce').astype('Int64')
df_ppm['N° Cliente'] = pd.to_numeric(df_ppm['N° Cliente'], errors='coerce').astype('Int64')

# Merge the dataframes
df_merged = pd.merge(df_pacs, df_ppm, left_on='Atendimento', right_on='N° Cliente', how='inner')

# Select columns
selected_columns = ['Request No', 'N° Cliente', 'Abertura', 'Data da última atualização', 'Encerramento', 'Passo', 'Situação', 'Prioridade_x', 'Prioridade_y', 'Assignado para', 'Líder Técnico', 'Módulo', 'Nivel', 'Atraso na resposta', 'Atraso no serviço', 'Solução', 'Descrição']
df_merged = df_merged[selected_columns]

# Rename columns
rename_map = {'Passo': 'Status_ppm', 'Situação': 'Status_pacs', 'Prioridade_x': 'Prioridade_pacs', 'Prioridade_y': 'Prioridade_ppm'}
df_merged = df_merged.rename(columns=rename_map)

# Create 'Status Comparado' column
def compare_status(row):
    if (row['Status_ppm'] == 'Acknowledge' and row['Status_pacs'] == 'Em atendimento') or \
       (row['Status_ppm'] == 'On Hold'     and row['Status_pacs'] == 'Suspenso') or \
       (row['Status_ppm'] == 'Closed'      and row['Status_pacs'] == 'Cancelado') or \
       (row['Status_ppm'] == 'Closed'      and row['Status_pacs'] == 'Encerrado') or \
       (row['Status_ppm'] == 'Cancelled'   and row['Status_pacs'] == 'Encerrado') or \
       (row['Status_ppm'] == 'Cancelled'   and row['Status_pacs'] == 'Cancelado') or \
       (row['Status_ppm'] == 'Closed'      and row['Status_pacs'] == 'Aguardando confirmação de encerramento') or \
       (row['Status_ppm'] == 'Closed'      and row['Status_pacs'] == 'Em atendimento') or \
       (row['Status_ppm'] == 'Working - Cadastro' and row['Status_pacs'] == 'Em atendimento') or \
       (row['Status_ppm'] == 'Working - M' and row['Status_pacs'] == 'Em atendimento') or \
       (row['Status_ppm'] == 'Analyse - M' and row['Status_pacs'] == 'Em atendimento') or \
       (row['Status_ppm'] == 'Analyse - Sup' and row['Status_pacs'] == 'Em atendimento') or \
       (row['Status_ppm'] == row['Status_pacs']):
        return 'IGUAL'
    else:
        return 'DIFERENTE'

df_merged['Status Comparado'] = df_merged.apply(compare_status, axis=1)

# Insert 'Status Comparado' after 'Status_pacs'
cols = df_merged.columns.tolist()
status_pacs_index = cols.index('Status_pacs')
if 'Status Comparado' in cols:
    cols.remove('Status Comparado')
cols.insert(status_pacs_index + 1, 'Status Comparado')
df_merged = df_merged[cols]

# Save the result to an Excel file
brasilia_timezone = timezone(timedelta(hours=-3))
current_time_brasilia = datetime.now(brasilia_timezone)
timestamp = current_time_brasilia.strftime("%Y%m%d_%H%M%S")
filename = f"Analise_PPM_PACS_{timestamp}.xlsx"

df_merged.to_excel(filename, index=False)

display(df_merged.head())
print(f"File saved successfully as {filename}")

/usr/local/lib/python3.12/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Parsing dates in %d/%m/%Y - %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  cast_date_col = pd.to_datetime(column, errors="coerce")


,Request No,N° Cliente,Abertura,Data da última atualização,Encerramento,Status_ppm,Status_pacs,Status Comparado,Prioridade_pacs,Prioridade_ppm,Assignado para,Líder Técnico,Módulo,Nivel,Atraso na resposta,Atraso no serviço,Solução,Descrição
0,6208842,353831,22/09/2025 - 14:19,22/09/2025 - 14:27,NaN,Acknowledge,Em atendimento,IGUAL,4. Baixa,Baixa,AMS_MM AMS_MM,Giselle Lourenco de Oliveira Stefani,MM,N2,Não,Não,22/09/2025 14:26 Foi criado o ticket PPM N 620...,/N/ACHE/150101MM0003 liberar acesso a transaçã...
1,6208497,353818,22/09/2025 - 13:37,22/09/2025 - 13:52,NaN,Acknowledge,Em atendimento,IGUAL,2. Alta,Muito Alta,AMS_SD AMS_SD,Silvia Maria Martins Custodio,SD,N2,Não,Não,22/09/2025 13:41 Foi criado o ticket PPM N 620...,Remessas ECC não teve integração S4 não criou ...
2,6208476,353793,22/09/2025 - 11:56,22/09/2025 - 12:06,NaN,Acknowledge,Em atendimento,IGUAL,3. Média,Media,AMS_MM AMS_MM,Giselle Lourenco de Oliveira Stefani,WM/EWM,N2,Não,Não,22/09/2025 12:06 Foi criado o ticket PPM N 620...,Erro no momento de realizar a SM no EWM.
3,6208076,353778,22/09/2025 - 11:18,22/09/2025 - 11:42,22/09/2025 - 11:42,Closed,Aguardando confirmação de encerramento,IGUAL,4. Baixa,Baixa,Felipe Camerlengo Fontes,Edileine Serrano,Concur,N2,Não,Não,22/09/2025 11:21 Foi criado o ticket PPM N 620...,Delegar o RDV da colaboradora desligada Lucian...
4,6208071,353707,22/09/2025 - 10:10,22/09/2025 - 11:24,22/09/2025 - 11:24,Closed,Encerrado,IGUAL,4. Baixa,Baixa,Felipe Camerlengo Fontes,Edileine Serrano,Concur,N2,Não,Não,22/09/2025 10:11 Foi criado o ticket PPM N 620...,Preciso que deleguem o acesso do ex colaborado...


File saved successfully as Analise_PPM_PACS_20250922_145800.xlsx
